# Simple Morphological Tree Examples

This notebook introduces the basic `mmcfilters` workflow on a small synthetic image and then repeats the same ideas on a real coin image.


## 1. Install the library

This notebook is a compact tour of the main `mmcfilters` workflow: build a morphological tree, compute attributes, filter connected components, and project the result back to an image.


In [ ]:
!pip install mmcfilters
!pip install bokeh
!pip install morphotreeviz

## 2. Import the library


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2 as cv


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)
import mmcfilters
import mtviz as viz
if not hasattr(viz, "show_level_sets"):
    viz.show_level_sets = getattr(viz, "showLevelSets", lambda *args, **kwargs: None)
from bokeh.io import output_notebook, show
from bokeh.layouts import column, row
output_notebook()


def create_component_tree(image, is_maxtree, radius=1.5):
    if is_maxtree:
        return mmcfilters.MorphologicalTreeFactory.createMaxTree(image, radius=radius)
    return mmcfilters.MorphologicalTreeFactory.createMinTree(image, radius=radius)


def print_tree_with_attribute(attribute_type, attribute_by_node):
    return lambda tree, node_id: f"id:{node_id}, {attribute_type.name}: {attribute_by_node[node_id]}"


def show_component_tree(tree, image=None, label=None):
    label = label or (lambda tree, node_id: f"id:{node_id}")

    def walk(node_id, depth=0):
        print("  " * depth + label(tree, node_id))
        for child_id in tree.getChildren(node_id):
            walk(child_id, depth + 1)

    if image is not None:
        plt.figure(figsize=(5, 5))
        plt.imshow(image, cmap='gray', vmax=255, vmin=0)
        plt.axis('off')
        plt.show()

    walk(tree.getRoot())


def show_tree(tree, label=None):
    show_component_tree(tree, label=label)


def getPlotTree(tree, title):
    show_component_tree(tree)
    return None


## 3. Create a morphological tree from an input image

A component tree represents connected level sets as nodes. In this example, the synthetic image is small enough that the tree structure can be printed and compared with the pixel values.


In [ ]:
#input_image = load_grayscale("../dat/imgTeste.png")

input_image = np.array([
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203,203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203],
        [203,203, 78, 78,126,126,126,126,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203,203, 54, 54,203,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203,203, 54, 54, 54, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126,126,126, 78, 78, 78, 78, 78, 78, 78,203, 54, 54, 54, 80, 54,203],
        [203,203, 78, 78,126, 38, 38, 38,126, 78, 78, 78, 78, 78, 78, 78, 78, 78,203, 54, 80, 54, 54, 54,203],
        [203, 78, 78, 78,126, 38, 38,126,126, 78, 78, 78,203,203,203,203,203,203,203, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126,126, 78, 78,203,203,203,203,203,203,203,203, 54, 54, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 80, 54, 54, 54,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203, 54, 54, 54, 54, 54,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78,203,203,253,253,253,203,203,203, 54, 54, 54,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,203,126,126,126,126,203,203,203,203,203,203,203],
        [203, 78, 78, 78,126,126,126, 78, 78, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,126,126,126,126,126,126, 72,126,126,203,203,203],
        [203, 78, 78, 78, 78, 78, 78,161,161,161, 78, 78,203,126,126,126,126,126,126, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78,203,126,126,126,126,126, 72, 72, 72,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78,203,203,126,126,126, 72, 72, 72, 72, 72,126,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 90, 30,161, 78, 78,203,126,126, 72, 72, 72, 72, 72, 72, 72,126,203],
        [203, 78, 78, 78, 78, 78,161, 30, 30, 30,161, 78, 78,203,203,126,126,126,126,126,126,126,126,126,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,126,126,126,126,126,126,126,203,203],
        [203, 78, 78, 78, 78, 78,161,161,161,161,161, 78, 78, 78,203,203,203,203,126,126,126,126,203,203,203],
        [203,203, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78, 78,203,203,203,203,203,203,203,203,203,203],
        [203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203,203]
], dtype=np.uint8)
input_image = np.ascontiguousarray(input_image, dtype=np.uint8)
(num_rows, num_cols) = input_image.shape

is_max_tree = False
tree = create_component_tree(input_image, is_max_tree)
viz.show_level_sets(input_image)


In [ ]:
show_component_tree(tree, image=input_image)

## 4. Available attributes

Attributes summarize geometric, radiometric, or topological properties of each node. Listing the available attributes is a good first step before choosing a filtering criterion.


In [ ]:
attribute_enum = type(mmcfilters.Attribute.AREA)
describe = {
    attribute_name: mmcfilters.Attribute.describe(attribute_value)
    for attribute_name in dir(mmcfilters.Attribute)
    if attribute_name.isupper()
    for attribute_value in [getattr(mmcfilters.Attribute, attribute_name)]
    if isinstance(attribute_value, attribute_enum)
}

df = pd.DataFrame(describe.items(), columns=['Attribute type', 'Description'])
df.style.set_caption("<H3><b>Available attributes</b></H3>")

## 5. Compute a single attribute

`computeSingleAttribute` returns one value per tree node. The gray-height attribute is used here because it is easy to inspect on the printed tree.


In [ ]:
attribute_type = mmcfilters.Attribute.GRAY_HEIGHT
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, attribute_type)

print(f"{attribute_type.name} (numpy): {attribute_by_node}")

In [ ]:
show_tree(tree, print_tree_with_attribute(attribute_type, attribute_by_node))

## 6. Compute multiple attributes

`computeAttributes` evaluates several attributes together and returns a node-by-attribute table. This is useful when a filter or analysis combines multiple criteria.


In [ ]:
attribute_indices, attribute_matrix = mmcfilters.Attribute.computeAttributes(tree, [mmcfilters.Attribute.AREA, mmcfilters.Attribute.GRAY_HEIGHT, mmcfilters.Attribute.VOLUME, mmcfilters.Attribute.RELATIVE_VOLUME])
#print(attribute_matrix[:, attribute_indices['GRAY_HEIGHT']])
pd.DataFrame(attribute_matrix, columns=attribute_indices, index=pd.Index(range(len(attribute_matrix)), name="NodeID"))

## 7. Attribute filtering

Attribute filters remove or preserve nodes according to a criterion and then reconstruct an image from the modified tree. This example uses the subtractive rule with the gray-height values.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT)
threshold = 80

attribute_filter = mmcfilters.AttributeFilters(tree)
filtered_image = attribute_filter.filteringSubtractiveRule(attribute_by_node > threshold)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image.reshape(num_rows, num_cols), cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

## 8. Extract extinction values

Extinction values rank extrema by the importance of the attribute that disappears when components merge. They are often more stable than a fixed threshold on raw attribute values.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT) # the attribute must be increasing

In [ ]:
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node)
for leaf_id, cutoff_node_id, extinction in extinction_values.getExtinctionValues():
    print("Regional extremum (leaf):", leaf_id)
    print("Extinction value: ", extinction)
    print("Persistence node where the regional extremum still exists:", cutoff_node_id)
    print("The regional extremum disappears at:", tree.getNodeParent(cutoff_node_id))
    print()

show_component_tree(tree, image=input_image)


## 9. Filter by extinction values

Here the filter keeps only the most relevant extrema according to the extinction ranking. This provides a compact way to control the number of preserved structures.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleAttribute(tree, mmcfilters.Attribute.GRAY_HEIGHT)
attribute_filter = mmcfilters.AttributeFilters(tree)

num_leaves_to_keep = 3 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image = attribute_filter.filteringByExtinction(attribute_by_node, num_leaves_to_keep)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 3 regional extrema')

## 10. Work with the coin image

The same workflow is applied to a real image: direct attribute filtering, extinction-value filtering, and saliency-map construction.


In [ ]:
# 1. Filtering

from skimage import data, img_as_float
input_image = np.ascontiguousarray(data.coins(), dtype=np.uint8)
(num_rows, num_cols) = input_image.shape

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)

attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
threshold = 500

attribute_filter = mmcfilters.AttributeFilters(tree)
filtered_image = attribute_filter.filteringSubtractiveRule(attribute_by_node > threshold)

plt.subplot(1,2, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,2, 2)
plt.imshow(filtered_image, cmap='gray', vmax=255, vmin=0)
plt.title('attribute filter')

In [ ]:
# 2. Filtering by extinction values

input_image = filtered_image

is_max_tree = True
tree = create_component_tree(input_image, is_max_tree)
attribute_filter = mmcfilters.AttributeFilters(tree)
attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)

num_leaves_to_keep = 6 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_6 = attribute_filter.filteringByExtinction(attribute_by_node, num_leaves_to_keep)

num_leaves_to_keep = 24 # keep num_leaves_to_keep leaves with the highest extinction values
filtered_image_24 = attribute_filter.filteringByExtinction(attribute_by_node, num_leaves_to_keep)

plt.figure(figsize=(15, 5))
plt.subplot(1,3, 1)
plt.imshow(input_image, cmap='gray', vmax=255, vmin=0)
plt.title('input')

plt.subplot(1,3, 2)
plt.imshow(filtered_image_6, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 6 regional extrema')

plt.subplot(1,3,3)
plt.imshow(filtered_image_24, cmap='gray', vmax=255, vmin=0)
plt.title('keeping 24 regional extrema')

In [ ]:
# 3. Creating a saliency map

attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.numLeafNodes * 1) # 5% of the regional extrema

contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_cols), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in sorted(extinction_values.getExtinctionValues(), key=lambda x: x[2], reverse=True)[:num_leaves_to_keep]:
    for p in contours.getContour(cutoff_node_id):
        contour_image[p]= circularity[cutoff_node_id]
        #contour_image[p]= importance
    importance -= 1


plt.figure(figsize=(5, 5))
plt.imshow(contour_image.reshape(num_rows, num_cols), cmap='Grays')
plt.axis('off')
plt.show()


## 11. Combine shape criteria on contours

This final section uses area and circularity to build contour-based saliency maps. The approach is limited to increasing attributes, so the chosen criteria should be checked before reuse.


In [ ]:
attribute_by_node = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.AREA)
circularity = mmcfilters.Attribute.computeSingleTopologyAttribute(tree, mmcfilters.Attribute.CIRCULARITY)
extinction_values = mmcfilters.ExtinctionValues(tree, attribute_by_node) # the attribute must be increasing
num_leaves_to_keep = int(tree.numLeafNodes * 1)

contours = mmcfilters.ContourComputation.extraction(tree)

contour_image = np.zeros((num_rows*num_cols), dtype=np.float32)
importance = num_leaves_to_keep
for leaf_id, cutoff_node_id, extinction in sorted(extinction_values.getExtinctionValues(), key=lambda x: x[2], reverse=True)[:num_leaves_to_keep]:
    for p in contours.getContour(cutoff_node_id):
        contour_image[p]=circularity[cutoff_node_id]
    importance -= 1



plt.figure(figsize=(15, 5))
plt.subplot(1,2, 1)
plt.imshow(contour_image.reshape(num_rows, num_cols), cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the circularity')

plt.subplot(1,2, 2)
saliency_map = extinction_values.saliencyMap(num_leaves_to_keep)
plt.imshow(saliency_map, cmap='Grays')
plt.axis('off')
plt.title('saliency map: importance is the extinction values')
plt.show()
